# Worm Moebius

In this notebook, we implement the worm algorithm for the Moebius code for $d = 2 p$ with $p$ odd prime. 

In [1]:
import os
from functools import partial
from jax.typing import ArrayLike
from jax import Array
from typing import Tuple, Dict
import jax.numpy as jnp
import numpy as np
from coherentinfo.moebius_two_odd_prime import MoebiusCodeTwoOddPrime
from coherentinfo.errormodel import ErrorModelLindbladTwoOddPrime
from coherentinfo.worm import (
    worm_step,
    run_worm
)
import jax
n_cpus = os.cpu_count()
print("Number of CPUs available: {}".format(n_cpus))
jax.config.update('jax_num_cpu_devices', n_cpus)
# jax.config.update('jax_num_cpu_devices', 16)

Number of CPUs available: 16


In [2]:
length = 11  # 5
width = 11  # 5
p = 3
d = 2 * p
moebius_code = MoebiusCodeTwoOddPrime(length=length, width=width, d=d)
h_z = moebius_code.h_z
h_x = moebius_code.h_x
h_z_mod_2 = moebius_code.h_z_mod_2
h_z_mod_p = moebius_code.h_z_mod_p
h_x_mod_2 = moebius_code.h_x_mod_2
h_x_mod_p = moebius_code.h_x_mod_p
logical_x = moebius_code.logical_x
logical_z = moebius_code.logical_z
num_plaquette, num_edges = h_x.shape
gamma_t = 0.3
em_lindblad = ErrorModelLindbladTwoOddPrime(
    moebius_code.num_edges, d=d, gamma_t=gamma_t
)

# error_master_key = jax.random.PRNGKey(42)
# initial_error = em_lindblad.generate_random_error(error_key)
# initial_error_mod_2 = jnp.mod(initial_error, 2)
# initial_error_mod_p = jnp.mod(initial_error, p)
# # Here we consider the full syndrome including the plaquette
# # we usually remove because of the constraint as this simplified the
# # coding of the worm algorithm. In fact, in this the syndromes will
# # always be annihilated in pairs, and the total number of syndromes is
# # always even as one can check numerically.
# syndrome = moebius_code.get_plaquette_syndrome(initial_error)
# syndrome_mod_2 = jnp.mod(syndrome, 2)
# syndrome_mod_p = jnp.mod(syndrome, p)
# num_plaquette, num_edges = h_x.shape

In [7]:
master_worm_key = jax.random.PRNGKey(144)
num_worms = 10 * n_cpus
worm_keys = jax.random.split(master_worm_key, num_worms)

error_master_key = jax.random.PRNGKey(42)
num_samples = 10 * n_cpus
error_keys = jax.random.split(master_worm_key, num_worms)


def generate_initial_worm_errors(
    key: ArrayLike,
    error_model: ErrorModelLindbladTwoOddPrime
) -> ArrayLike:
    initial_error = error_model.generate_random_error(key)
    initial_error_mod_2 = jnp.mod(initial_error, 2)
    initial_error_mod_p = jnp.mod(initial_error, p)
    initial_worm_error = jnp.vstack((initial_error_mod_2, initial_error_mod_p))
    return initial_worm_error


initial_worm_errors = jax.vmap(
    generate_initial_worm_errors, in_axes=(0, None))(error_keys, em_lindblad)

In [8]:
# master_worm_key = jax.random.PRNGKey(144)
# num_worms = 100
# worm_keys = jax.random.split(master_worm_key, num_worms)

# error_master_key = jax.random.PRNGKey(42)
# num_samples = 100
# worm_keys = jax.random.split(master_worm_key, num_worms)
# initial_error = em_lindblad.generate_random_error(error_key)

max_steps = 100
initial_worm_state = {}
# worm_error = jnp.vstack(
#     (initial_error_mod_2, initial_error_mod_p))
initial_head = 18
initial_worm_state["head"] = initial_head
initial_worm_state["tail"] = initial_head
initial_worm_state["worm_success"] = False
initial_worm_state["accepted_moves"] = 0
initial_worm_state["attempted_moves"] = 0

run_worm_partial = partial(
    run_worm,
    initial_worm_state=initial_worm_state,
    h_error_mod_p=h_z_mod_p,
    h_mod_p=h_x_mod_p,
    error_model=em_lindblad,
    max_worm_steps=max_steps
)

# First over keys
run_worm_vmap = jax.vmap(run_worm_partial, in_axes=(None, 0))
# Then over initial errors
run_worm_vmap = jax.vmap(run_worm_vmap, in_axes=(0, None))
run_worm_jit = jax.jit(run_worm_vmap)
new_worm_state = run_worm_jit(initial_worm_errors, worm_keys)

# index = 0
# print("Number of accepted moves: {}".format(new_worm_state["accepted_moves"][index]))
# print("Number of attempted moves: {}".format(
#     new_worm_state["attempted_moves"][index]))

In [9]:
index = 28
print("Number of accepted moves: {}".format(
    new_worm_state["accepted_moves"][index]))
print("Number of attempted moves: {}".format(
    new_worm_state["attempted_moves"][index]))
print("Success: {}".format(
    new_worm_state["worm_success"][index]))

Number of accepted moves: [ 4  7  2  4  2  2 12  4  2  5 11  4  3  3  7  1  3  2  8  1  6  2  2  9
  5  4 14  2  1  2  3  4  5  3  1  2  4  3  2  9 10  3  4  4  3  5 10  5
  3  6  3  6  6  3  5  7  4  4  3 10  3  7  5  2  3  5  2  2  2  2  6  5
  6  7  2  2  7 13  8 10  3  5  7  2 11  2  2 10 14  5  6 14  2  2  5  7
  3  3  7  2  6  4  4  2  1  8  2  3  7  5  4  3  9  7  7  6  2  3  4  6
  5  5  5  3  3  7  2  5  1  8  8  7  5  3  2  2  2  3  5  3  6  4  2  2
  4  4  7  8  8  1  3  2  3  3  4  7 14  5  4 12]
Number of attempted moves: [100 100  10 100   5  10 100 100  36 100 100  12 100 100 100 100 100 100
 100 100 100  21   7 100 100  61 100   2 100  52 100  29 100 100 100  32
  42 100  17 100 100 100 100 100 100 100  77 100 100 100 100 100  74 100
 100 100  14  13 100  84 100 100 100  22 100 100  25 100   6  22  32 100
 100 100 100  29 100 100 100 100 100 100 100  15 100   5   9  65 100 100
 100 100  16   3 100 100 100 100 100 100  39 100  92   2 100 100  16 100
 100 100  21 100 100 

In [9]:
print(new_worm_state["head"])
print(initial_worm_state["head"])

[[18 18 18 ... 18 18 18]
 [18 18 18 ... 18 18 18]
 [18 18 18 ... 18 18 18]
 ...
 [18 18 18 ... 18 18 18]
 [18 18 18 ... 18 18 18]
 [18 18 18 ... 18 18 18]]
18


In [15]:
initial_worm_errors[0][0]

Array([0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1,
       1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0,
       1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0,
       0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
       1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0], dtype=int32)

In [16]:
index_a = 0
index_b = 0
print(jnp.max(
    jnp.abs(jnp.mod(new_worm_state["worm_error"][index_a, index_b][0, :] - initial_worm_errors[index_a][0, :], 2))))

1


In [17]:
print(jnp.max(
    jnp.abs(jnp.mod(new_worm_state["worm_error"][index_a, index_b][1, :] - initial_worm_errors[index_a][1, :], p))))

2


In [ ]:
syndrome_mod_2 = jnp.mod(h_x_mod_2 @ initial_worm_errors[index_a][0, :], 2)
new_syndrome_mod_2 = jnp.mod(
    h_x_mod_2 @ new_worm_state["worm_error"][index_a, index_b][0, :], 2)

jnp.mod(new_syndrome_mod_2 - syndrome_mod_2, 2)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [ ]:
syndrome_mod_p = jnp.mod(h_x_mod_p @ initial_worm_errors[index_a][1, :], p)
new_syndrome_mod_p = jnp.mod(
    h_x_mod_p @ new_worm_state["worm_error"][index_a, index_b][1, :], p)

jnp.mod(new_syndrome_mod_p - syndrome_mod_p, p)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)